# 05 — Submit ALL requested monthly Hakaluki map exports to Google Earth Engine

This notebook creates resumable Google Drive export tasks and logs each task ID/state. It **does not** fabricate images or claim a queued task is complete. Change `DRY_RUN=False` when ready. A batch is limited to avoid hitting quotas. All maps inherit a **user-provided boundary that still needs source/geometry validation**.

**Important:** TROPOMI is atmospheric XCH₄, **not local methane flux**. SAR low-VV water screens are not accuracy-validated flood extents. Dynamic World class 3 is classified flooded vegetation, not an independently validated flood duration.'
 

## A. Initialize YOUR Windows project and YOUR GEE Cloud project

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path(r"F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project")
assert (ROOT / 'config' / 'settings.json').exists(), f"Wrong project folder: {ROOT}"
sys.path.insert(0, str(ROOT))
from src.ee_release_exports import create_environment, submit_batch, refresh_status, DATASETS
ENV = create_environment(ROOT)
print('Project:', ROOT)
print('EE Cloud project:', ENV['cfg']['earth_engine_cloud_project'])
print('Boundary file:', ENV['boundary_path'])
print('Sentinel-1 pilot pass/orbit:', ENV['sar']['orbit_pass'], ENV['sar']['relative_orbit'])


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Project: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project
EE Cloud project: axial-paratext-491911-b6
Boundary file: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\data\boundary\hakaluki_analysis_candidate.geojson
Sentinel-1 pilot pass/orbit: ASCENDING 41


## B. Select years/months and map layer types

Recommended first run: 2022, March–September. To process all of 2019–2025 use `YEARS=list(range(2019,2026))`, `MONTHS=list(range(1,13))`. This can mean hundreds of export jobs, so submit in small batches. All source products have different spatial supports; task scale does not increase their true resolution.

In [4]:

# Hakaluki Haor: initial spatial map exports

YEARS = [2022]
MONTHS = [3, 6, 10]

LAYER_TYPES = [
    "SAR_water",
    "S2_NDVI",
    "S2_MNDWI",
    "DW_class",
    "CHIRPS_rain",
    "ERA5_temp"
]

MAX_NEW_TASKS = 6
DRY_RUN = False



## C. Preview or submit the next unfinished batch

In [8]:
RESULT = submit_batch(
    ENV, ROOT, years=YEARS, months=MONTHS, types=LAYER_TYPES,
    max_new_tasks=MAX_NEW_TASKS, dry_run=DRY_RUN
)
print(RESULT)


Planned jobs: 18 | already recorded: 12 | pending: 6
Submitted HAKALUKI_2022_10_SAR_water [READY]
Submitted HAKALUKI_2022_10_S2_NDVI [READY]
Submitted HAKALUKI_2022_10_S2_MNDWI [READY]
Submitted HAKALUKI_2022_10_DW_class [READY]
Submitted HAKALUKI_2022_10_CHIRPS_rain [READY]
Submitted HAKALUKI_2022_10_ERA5_temp [READY]
{'dry_run': False, 'n_new_tasks': 6, 'n_planned': 18, 'n_recorded': 18, 'manifest_csv': 'F:\\Wetland Test\\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\\results\\tables\\ee_export_task_manifest.csv', 'note': 'A task may still be QUEUED/RUNNING/FAILED. Download successful GeoTIFFs from Google Drive.'}


## D. Inspect submitted task states

A submitted task may still be RUNNING, FAILED, CANCELLED, or COMPLETED. The manifest is an audit log **not** a receipt for downloaded data. In case of errors, inspect the Earth Engine Tasks interface and the Google Drive destination. A missing-scene month may have no task and should not be treated as zero water/methane.

In [9]:
# Only refresh tasks when there is a submitted manifest.
TASKS = refresh_status(ROOT)
if TASKS.empty:
    print('No tasks recorded yet. Set DRY_RUN=False, run Section C, then return here.')
else:
    display(TASKS.tail(15))
    print('Task states:', TASKS['task_state'].value_counts(dropna=False).to_dict())
    print('Manifest:', ROOT / 'results' / 'tables' / 'ee_export_task_manifest.csv')


,export_name,year,month,source,source_type,unit,scale_m_approx,drive_folder,expected_filename,task_id,task_state,submitted_utc,notes
3,HAKALUKI_2022_03_DW_class,2022,3,Dynamic World label mode,DW_class,categorical 0..8,30,Hakaluki_Haor_Study,HAKALUKI_2022_03_DW_class.tif,I7WM6HSGMQPNGU623OOYYQJQ,COMPLETED,2026-10-13T18:49:48.830868+00:00,"{""dw_scenes"": 20}"
4,HAKALUKI_2022_03_CHIRPS_rain,2022,3,CHIRPS daily precipitation sum,CHIRPS_rain,mm/month,5500,Hakaluki_Haor_Study,HAKALUKI_2022_03_CHIRPS_rain.tif,WGXXROAICQEPN7CP3VNX7GYS,COMPLETED,2026-10-13T18:49:52.066686+00:00,{}
5,HAKALUKI_2022_03_ERA5_temp,2022,3,ERA5-Land monthly 2m mean temperature,ERA5_temp,degrees C,11000,Hakaluki_Haor_Study,HAKALUKI_2022_03_ERA5_temp.tif,BOGGRBBNSIE37D4QBUDNPA4W,COMPLETED,2026-10-13T18:49:55.672551+00:00,{}
6,HAKALUKI_2022_06_SAR_water,2022,6,Sentinel-1 GRD low-VV Otsu water screening,SAR_water,0/1 preliminary water screen,30,Hakaluki_Haor_Study,HAKALUKI_2022_06_SAR_water.tif,BVOYYWWQ2ZQD7ZPDFDOBAKM6,COMPLETED,2026-10-13T18:52:18.964503+00:00,"{""threshold_db"": -15.20579678213733, ""s1_scene..."
7,HAKALUKI_2022_06_S2_NDVI,2022,6,Sentinel-2 SR NDVI,S2_NDVI,unitless NDVI,20,Hakaluki_Haor_Study,HAKALUKI_2022_06_S2_NDVI.tif,Y642EVVJHHY6BYSZDOWQAHTQ,COMPLETED,2026-10-13T18:52:22.235463+00:00,"{""s2_scenes"": 24}"
8,HAKALUKI_2022_06_S2_MNDWI,2022,6,Sentinel-2 SR MNDWI,S2_MNDWI,unitless MNDWI,20,Hakaluki_Haor_Study,HAKALUKI_2022_06_S2_MNDWI.tif,S3S457W4YC6ERHYL72ACV33F,COMPLETED,2026-10-13T18:52:25.525738+00:00,"{""s2_scenes"": 24}"
9,HAKALUKI_2022_06_DW_class,2022,6,Dynamic World label mode,DW_class,categorical 0..8,30,Hakaluki_Haor_Study,HAKALUKI_2022_06_DW_class.tif,QP4GDOEK2DEQ5B4YNPLZC66C,COMPLETED,2026-10-13T18:52:30.089304+00:00,"{""dw_scenes"": 1}"
10,HAKALUKI_2022_06_CHIRPS_rain,2022,6,CHIRPS daily precipitation sum,CHIRPS_rain,mm/month,5500,Hakaluki_Haor_Study,HAKALUKI_2022_06_CHIRPS_rain.tif,CBBA73Z5COBGBOGBJGPY4DOZ,COMPLETED,2026-10-13T18:52:33.510778+00:00,{}
11,HAKALUKI_2022_06_ERA5_temp,2022,6,ERA5-Land monthly 2m mean temperature,ERA5_temp,degrees C,11000,Hakaluki_Haor_Study,HAKALUKI_2022_06_ERA5_temp.tif,5VVSS65JQLC52WMQDMCJNRFX,COMPLETED,2026-10-13T18:52:38.339129+00:00,{}
12,HAKALUKI_2022_10_SAR_water,2022,10,Sentinel-1 GRD low-VV Otsu water screening,SAR_water,0/1 preliminary water screen,30,Hakaluki_Haor_Study,HAKALUKI_2022_10_SAR_water.tif,2WYT7G5UMIERZSKBRFS6FC4P,COMPLETED,2026-10-13T18:58:40.377168+00:00,"{""threshold_db"": -16.404300004789242, ""s1_scen..."


Task states: {'COMPLETED': 18}
Manifest: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\results\tables\ee_export_task_manifest.csv


## E. Download and publish ALL completed outputs

1. When tasks are **COMPLETED**, open Google Drive folder `Hakaluki_Haor_Study`.
2. Download every GeoTIFF into `results/rasters/` under your Windows project. Keep original filenames.
3. Run Notebook **06 — GitHub portfolio release** to convert TIFFs to map PNGs, plot real time series, copy all analysis CSVs, and generate `README.md`, `index.html`, and SHA-256 inventories.
4. Run Notebook 05 again to submit the next batch. Your log skips already-submitted export names.

The original GeoTIFFs stay on disk. GitHub gets web-sized maps and data tables, not hundreds of very large rasters.
